<a href="https://colab.research.google.com/github/Rohita-G/NLP_CLASS/blob/main/NLP_HW_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Q1 — Regex

In [9]:
import re

text = """
My ZIP codes are 64093, 64093-1234, and 64093 5678.
These are invalid: 640931234 and 123456.
"""

pattern = r'\b\d{5}(?:[- ]\d{4})?\b'

print(re.findall(pattern, text))

['64093', '64093-1234', '64093 5678']


In [10]:
text = "Hello world don't use state-of-the-art methods. Python is useful."

pattern = r"\b(?![A-Z])[A-Za-z]+(?:['-][A-Za-z]+)*\b"

print(re.findall(pattern, text))

['world', "don't", 'use', 'state-of-the-art', 'methods', 'is', 'useful']


In [11]:
text = "Numbers: 12, -45, +123, 1,234, 3.14, -2.5, 1.23e-4, +6.02E23"

pattern = r'(?<!\w)[+-]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?(?:[eE][+-]?\d+)?(?!\w)'

print(re.findall(pattern, text))

['12', '-45', '+123', '1,234', '3.14', '-2.5', '1.23e-4', '+6.02E23']


In [12]:
text = "Email, email, e-mail, e mail, and e–mail are all valid spellings."

pattern = r'\be(?:[ -–])?mail\b'

print(re.findall(pattern, text, re.IGNORECASE))

['Email', 'email', 'e-mail', 'e mail', 'e–mail']


In [13]:
text = "go goo gooo goooo! go? go, stop going goodbye."

pattern = r'\bgo+\b[!.,?]?'

print(re.findall(pattern, text))

['go', 'goo', 'gooo', 'goooo!', 'go?', 'go,']


In [24]:
text = """How are you?
Are you coming?
This is a statement.
Really?"""

pattern = r'^.*\?\s*[\)"’\]]*\s*$'

print(re.findall(pattern, text, re.MULTILINE))

['How are you?', 'Are you coming?', 'Really?']


#Q2.1 — Manual BPE on Toy Corpus

In [15]:
from collections import Counter

# --------------------------------------------------
# Create the toy corpus given in the assignment.
# Each word will later receive "_" as its
# end-of-word marker.
# --------------------------------------------------

corpus = """
low low low low low
lowest lowest
newer newer newer newer newer newer
wider wider wider
new new
"""


# --------------------------------------------------
# Add the end-of-word marker "_" to every word.
# The marker allows BPE to distinguish the end
# of a word from the same characters inside a word.
# --------------------------------------------------

words = [list(word) + ["_"] for word in corpus.split()]


# --------------------------------------------------
# Count all adjacent pairs of tokens (bigrams).
# For example, "low_" becomes:
# (l, o), (o, w), and (w, _)
# --------------------------------------------------

def get_bigram_counts(words):
    counts = Counter()

    for word in words:
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            counts[pair] += 1

    return counts


# --------------------------------------------------
# Merge a selected bigram into one new token.
# For example:
# (e, r) -> "er"
#
# The function goes through every word and replaces
# each occurrence of the selected pair.
# --------------------------------------------------

def merge_pair(words, pair):
    new_words = []

    for word in words:
        new_word = []
        i = 0

        while i < len(word):

            # Check whether the current token and
            # the next token form the pair we want to merge.
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:

                # Combine the two tokens into one token.
                new_word.append(word[i] + word[i + 1])

                # Skip both original tokens because
                # they have now been replaced by one token.
                i += 2

            else:
                # Keep the current token unchanged.
                new_word.append(word[i])
                i += 1

        new_words.append(new_word)

    return new_words


# --------------------------------------------------
# Create the initial vocabulary.
# The vocabulary contains every unique character
# plus the end-of-word marker "_".
# --------------------------------------------------

vocab = set()

for word in words:
    vocab.update(word)

print("INITIAL VOCABULARY:")
print(vocab)
print("Vocabulary size:", len(vocab))


# --------------------------------------------------
# Calculate and display the initial bigram counts.
# These counts are used to determine which pair
# should be merged first.
# --------------------------------------------------

bigram_counts = get_bigram_counts(words)

print("\nINITIAL BIGRAM COUNTS:")

for pair, count in bigram_counts.items():
    print(pair, "=", count)


# --------------------------------------------------
# Perform the first three BPE merges.
#
# At each step:
# 1. Find the most frequent bigram.
# 2. Merge that bigram into a new token.
# 3. Add the new token to the vocabulary.
# 4. Display the updated corpus and vocabulary.
# --------------------------------------------------

for step in range(1, 4):

    # Recalculate bigram frequencies after each merge.
    bigram_counts = get_bigram_counts(words)

    # Select the most frequent bigram.
    most_frequent_pair, count = bigram_counts.most_common(1)[0]

    # Create the new token by joining the two tokens.
    new_token = most_frequent_pair[0] + most_frequent_pair[1]

    # Replace the selected pair throughout the corpus.
    words = merge_pair(words, most_frequent_pair)

    # Add the newly created token to the vocabulary.
    vocab.add(new_token)

    print("\n--------------------------------")
    print("STEP", step)
    print("--------------------------------")

    # Display the pair selected for this merge.
    print("Most frequent pair:", most_frequent_pair)
    print("Count:", count)

    # Display the actual BPE merge operation.
    print("Merge:",
          most_frequent_pair[0],
          "+",
          most_frequent_pair[1],
          "->",
          new_token)

    # Display the updated corpus.
    print("\nUpdated corpus:")

    # Count identical tokenized words so that
    # repeated words can be displayed with their frequency.
    word_counts = Counter(tuple(word) for word in words)

    for word, frequency in word_counts.items():
        print(" ".join(word), ":", frequency)

    # Display the new token created by this merge.
    print("\nNew token:", new_token)

    # Display the vocabulary after the merge.
    print("Updated vocabulary:")
    print(vocab)

    # Display the new vocabulary size.
    print("Vocabulary size:", len(vocab))

INITIAL VOCABULARY:
{'r', '_', 'e', 'l', 'd', 'o', 't', 'n', 'w', 'i', 's'}
Vocabulary size: 11

INITIAL BIGRAM COUNTS:
('l', 'o') = 7
('o', 'w') = 7
('w', '_') = 7
('w', 'e') = 8
('e', 's') = 2
('s', 't') = 2
('t', '_') = 2
('n', 'e') = 8
('e', 'w') = 8
('e', 'r') = 9
('r', '_') = 9
('w', 'i') = 3
('i', 'd') = 3
('d', 'e') = 3

--------------------------------
STEP 1
--------------------------------
Most frequent pair: ('e', 'r')
Count: 9
Merge: e + r -> er

Updated corpus:
l o w _ : 5
l o w e s t _ : 2
n e w er _ : 6
w i d er _ : 3
n e w _ : 2

New token: er
Updated vocabulary:
{'r', '_', 'e', 'l', 'd', 'o', 't', 'n', 'w', 'i', 's', 'er'}
Vocabulary size: 12

--------------------------------
STEP 2
--------------------------------
Most frequent pair: ('er', '_')
Count: 9
Merge: er + _ -> er_

Updated corpus:
l o w _ : 5
l o w e s t _ : 2
n e w er_ : 6
w i d er_ : 3
n e w _ : 2

New token: er_
Updated vocabulary:
{'r', '_', 'e', 'l', 'd', 'o', 't', 'n', 'w', 'i', 's', 'er', 'er_'}
Voc

#Q2.2 — Code a Mini-BPE Learner

In [16]:

from collections import Counter

# --------------------------------------------------
# Toy corpus from the assignment.
# Each word will be represented as characters with
# "_" added as the end-of-word marker.
# --------------------------------------------------

corpus = """
low low low low low
lowest lowest
newer newer newer newer newer newer
wider wider wider wider
new new
"""

# Convert each word into a list of characters
# and add "_" to mark the end of the word.
words = [list(word) + ["_"] for word in corpus.split()]


# --------------------------------------------------
# Function to count all adjacent token pairs.
# These bigram counts are used to decide which pair
# should be merged next.
# --------------------------------------------------

def get_bigram_counts(words):
    counts = Counter()

    for word in words:
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            counts[pair] += 1

    return counts


# --------------------------------------------------
# Function to merge a selected pair of tokens.
# For example, (e, r) becomes the new token "er".
# --------------------------------------------------

def merge_pair(words, pair):
    new_words = []

    for word in words:
        new_word = []
        i = 0

        while i < len(word):

            # Check whether the current and next tokens
            # match the pair selected for merging.
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:

                # Combine the two tokens into one token.
                new_word.append(word[i] + word[i + 1])

                # Skip the two tokens that were merged.
                i += 2

            else:
                # Keep the current token unchanged.
                new_word.append(word[i])
                i += 1

        new_words.append(new_word)

    return new_words


# --------------------------------------------------
# Build the initial vocabulary from all characters
# and the end-of-word marker "_".
# --------------------------------------------------

vocab = set()

for word in words:
    vocab.update(word)

print("Initial vocabulary size:", len(vocab))


# --------------------------------------------------
# Learn BPE merges.
#
# At every step:
# 1. Count all adjacent pairs.
# 2. Select the most frequent pair.
# 3. Merge the pair.
# 4. Add the new token to the vocabulary.
# 5. Print the selected pair and vocabulary size.
# --------------------------------------------------

num_merges = 3

for step in range(1, num_merges + 1):

    # Count the frequency of every adjacent token pair.
    pair_counts = get_bigram_counts(words)

    # Select the most frequent pair.
    top_pair, count = pair_counts.most_common(1)[0]

    # Create the new token by joining the two tokens.
    new_token = top_pair[0] + top_pair[1]

    # Apply the merge to the entire corpus.
    words = merge_pair(words, top_pair)

    # Add the newly created token to the vocabulary.
    vocab.add(new_token)

    # Print the result of this BPE step.
    print(
        f"Step {step}: "
        f"Top pair = {top_pair}, "
        f"Count = {count}, "
        f"New token = {new_token}, "
        f"Vocabulary size = {len(vocab)}"
    )


Initial vocabulary size: 11
Step 1: Top pair = ('e', 'r'), Count = 10, New token = er, Vocabulary size = 12
Step 2: Top pair = ('er', '_'), Count = 10, New token = er_, Vocabulary size = 13
Step 3: Top pair = ('n', 'e'), Count = 8, New token = ne, Vocabulary size = 14


#Q2.2 — Word Segmentation
Using the BPE tokens learned in the previous steps:

new → ne + w + _

newer → ne + w + er_

lowest → l + o + w + e + s + t + _

widest → w + i + d + e + s + t + _

newestest → ne + w + e + s + t + e + s + t + _

#Q2.2 — OOV Explanation
BPE helps solve the OOV problem by breaking unknown words into smaller known parts. Even if the complete word was not seen before, its smaller subwords can still be used. For example, newer can be split into ne + w + er_. The token er_ can be useful because -er can occur as a suffix in English. This allows the model to handle new words without needing a separate token for every word. Overall, subword tokenization gives the model more flexibility with new or rare words.

#Q2.3 — Your Language / English(Q2.3 — Train BPE)
Paragraph:

I study data science at university. I learn Python and practice coding every day. My classes teach useful skills for data analysis and machine learning. I enjoy building small projects because projects help me understand new ideas. Learning regularly makes difficult topics easier and gives me confidence.


In [29]:
from collections import Counter

text = """I study data science at university.
I learn Python and practice coding every day.
My classes teach useful skills for data analysis and machine learning.
I enjoy building small projects because projects help me understand new ideas.
Learning regularly makes difficult topics easier and gives me confidence."""

# Convert text to lowercase
text = text.lower()

# Split into words
corpus = text.split()

# Add end-of-word marker
words = [list(word) + ["_"] for word in corpus]


def get_pair_counts(words):
    pairs = Counter()

    for word in words:
        for i in range(len(word) - 1):
            pairs[(word[i], word[i + 1])] += 1

    return pairs


def merge_pair(words, pair):
    new_words = []

    for word in words:
        new_word = []
        i = 0

        while i < len(word):

            if (
                i < len(word) - 1
                and word[i] == pair[0]
                and word[i + 1] == pair[1]
            ):
                new_word.append(word[i] + word[i + 1])
                i += 2
            else:
                new_word.append(word[i])
                i += 1

        new_words.append(new_word)

    return new_words


# Initial vocabulary
vocab = set(token for word in words for token in word)
initial_vocab_size = len(vocab)

merges = []

# Learn 30 merges
for step in range(30):

    pair_counts = get_pair_counts(words)

    if not pair_counts:
        break

    top_pair, count = pair_counts.most_common(1)[0]

    new_token = top_pair[0] + top_pair[1]

    words = merge_pair(words, top_pair)

    vocab.add(new_token)

    merges.append((top_pair, count, new_token))

print("Initial vocabulary size:", initial_vocab_size)
print("Final vocabulary size:", len(vocab))


print("\nFive most frequent merges:")
for pair, count, token in merges[:5]:
    print(pair, "→", token, "| Count:", count)


# Find longest tokens
tokens_sorted = sorted(vocab, key=lambda x: len(x), reverse=True)

print("\nFive longest tokens:")
for token in tokens_sorted[:5]:
    print(token, "| Length:", len(token))



Initial vocabulary size: 25
Final vocabulary size: 55

Five most frequent merges:
('s', '_') → s_ | Count: 8
('e', '_') → e_ | Count: 6
('e', 'a') → ea | Count: 6
('y', '_') → y_ | Count: 5
('.', '_') → ._ | Count: 5

Five longest tokens:
data_ | Length: 5
learn | Length: 5
and_ | Length: 4
data | Length: 4
lear | Length: 4


In [31]:
from collections import Counter

# 1. Short paragraph (4–6 sentences) in English
text = """I study data science at university. I learn Python and practice coding every day. My classes teach useful skills for data analysis and machine learning. I enjoy building small projects because projects help me understand new ideas. Learning regularly makes difficult topics easier and gives me confidence."""

# 2. Normalize and split into words
text = text.lower()
words_raw = text.split()

# 3. Add end-of-word marker "_"
words = [list(w) + ["_"] for w in words_raw]

# Helper: count adjacent token pairs
def get_pair_counts(words):
    pairs = Counter()
    for w in words:
        for i in range(len(w) - 1):
            pairs[(w[i], w[i+1])] += 1
    return pairs

# Helper: merge a selected pair in all words
def merge_pair(words, pair):
    new_words = []
    for w in words:
        nw = []
        i = 0
        while i < len(w):
            if i < len(w) - 1 and w[i] == pair[0] and w[i+1] == pair[1]:
                nw.append(w[i] + w[i+1])
                i += 2
            else:
                nw.append(w[i])
                i += 1
        new_words.append(nw)
    return new_words

# 4. Initial vocabulary
vocab = set(tok for w in words for tok in w)
initial_vocab_size = len(vocab)

merges = []

# 5. Learn at least 30 merges
for step in range(30):
    pair_counts = get_pair_counts(words)
    if not pair_counts:
        break
    top_pair, count = pair_counts.most_common(1)[0]
    new_token = top_pair[0] + top_pair[1]
    words = merge_pair(words, top_pair)
    vocab.add(new_token)
    merges.append((top_pair, count, new_token))

# 6. Show five most frequent merges
print("Initial vocabulary size:", initial_vocab_size)
print("\nFive most frequent merges:")
for i, (pair, count, token) in enumerate(merges[:5], 1):
    print(f"{i}. {pair} → {token} | Count: {count}")

# 7. Five longest subword tokens
tokens_sorted = sorted(vocab, key=len, reverse=True)
print("\nFive longest subword tokens:")
for tok in tokens_sorted[:5]:
    print(tok, "| Length:", len(tok))




Initial vocabulary size: 25

Five most frequent merges:
1. ('s', '_') → s_ | Count: 8
2. ('e', '_') → e_ | Count: 6
3. ('e', 'a') → ea | Count: 6
4. ('y', '_') → y_ | Count: 5
5. ('.', '_') → ._ | Count: 5

Five longest subword tokens:
data_ | Length: 5
learn | Length: 5
and_ | Length: 4
data | Length: 4
lear | Length: 4


#Q2.3 — Word Segmentation

Data → data_

Learning → learn + ing_

Practice → pr + a + ct + i + c + e_

University → u + n + i + v + er + si + t + y_

Regularly → r + e + g + ul + a + r + l + y_

# Q2.3 — Reflection
The BPE model learned different parts of words such as learn, ing_, and_, and data_. Some tokens are complete words, while others are parts of words or suffixes. One advantage of BPE is that it can handle rare and new words by breaking them into smaller known parts. Another advantage is that it reduces the number of unknown words. One disadvantage is that some subwords do not have a clear meaning by themselves. Another disadvantage is that some words can be split into many small pieces. Overall, BPE is useful because it can represent both common words and smaller parts of words.

# Q3 — Bayes Rule
###1
P(c) → The probability of a class before looking at the document. It is called the prior probability.

P(d | c) → The probability of seeing document d if it belongs to class c. It tells us how likely the document is for that class.

P(c | d) → The probability that the document belongs to class c after seeing the document. It is called the posterior probability.
###2
The denominator P(d) can be ignored when comparing classes because the document is the same for every class. Therefore, P(d) has the same value for every class. We only need to compare P(d | c)P(c) to determine which class has the highest probability.

# Q4 — Add-1 Smoothing
Given:

P(-) = 3/5
P(+) = 2/5
Vocabulary size V = 20
Total negative tokens = 14

1. Denominator:

Denominator = total tokens + V
            = 14 + 20
            = 34


2. P(predictable | -):

P(predictable | -)
= (2 + 1) / (14 + 20)
= 3/34
≈ 0.0882


3. P(fun | -):

P(fun | -)
= (0 + 1) / (14 + 20)
= 1/34
≈ 0.0294

# Q5 — Tokenization
### Q5.1 — Telugu Paragraph


నేను ప్రతిరోజూ కాలేజీకి వెళ్తాను. అక్కడ నేను కొత్త విషయాలు నేర్చుకుంటాను. నా స్నేహితులతో కలిసి చదువుతాను. సాయంత్రం ఇంటికి తిరిగి వస్తాను.


In [6]:
#Naive Tokenization
# Telugu paragraph

text = "నేను ప్రతిరోజూ కాలేజీకి వెళ్తాను. అక్కడ నేను కొత్త విషయాలు నేర్చుకుంటాను. నా స్నేహితులతో కలిసి చదువుతాను. సాయంత్రం ఇంటికి తిరిగి వస్తాను."

# Split wherever there is a space
naive_tokens = text.split()

print("Naive tokenization:")
print(naive_tokens)

Naive tokenization:
['నేను', 'ప్రతిరోజూ', 'కాలేజీకి', 'వెళ్తాను.', 'అక్కడ', 'నేను', 'కొత్త', 'విషయాలు', 'నేర్చుకుంటాను.', 'నా', 'స్నేహితులతో', 'కలిసి', 'చదువుతాను.', 'సాయంత్రం', 'ఇంటికి', 'తిరిగి', 'వస్తాను.']


Manually corrected tokens:

["నేను", "ప్రతి", "రోజు", "కాలేజీ", "కి", "వెళ్తాను", ".",
 "అక్కడ", "నేను", "కొత్త", "విషయాలు", "నేర్చుకుంటాను", ".",
 "నా", "స్నేహితులతో", "కలిసి", "చదువుతాను", ".",
 "సాయంత్రం", "ఇంటి", "కి", "తిరిగి", "వస్తాను", "."]


Differences:

ప్రతిరోజూ → ప్రతి + రోజు

కాలేజీకి → కాలేజీ + కి

వెళ్తాను. → వెళ్తాను + .

నేర్చుకుంటాను. → నేర్చుకుంటాను + .

చదువుతాను. → చదువుతాను + .

ఇంటికి → ఇంటి + కి

వస్తాను. → వస్తాను + .

##Compare with an NLP Tool

In [7]:
# Install Indic NLP Library
!pip install indic-nlp-library

In [8]:
from indicnlp.tokenize import indic_tokenize

text = "నేను ప్రతిరోజూ కాలేజీకి వెళ్తాను. అక్కడ నేను కొత్త విషయాలు నేర్చుకుంటాను. నా స్నేహితులతో కలిసి చదువుతాను. సాయంత్రం ఇంటికి తిరిగి వస్తాను."

tool_tokens = indic_tokenize.trivial_tokenize(text)

print("Tool tokenization:")
print(tool_tokens)

Tool tokenization:
['నేను', 'ప్రతిరోజూ', 'కాలేజీకి', 'వెళ్తాను', '.', 'అక్కడ', 'నేను', 'కొత్త', 'విషయాలు', 'నేర్చుకుంటాను', '.', 'నా', 'స్నేహితులతో', 'కలిసి', 'చదువుతాను', '.', 'సాయంత్రం', 'ఇంటికి', 'తిరిగి', 'వస్తాను', '.']


Comparison:

The tool keeps ప్రతిరోజూ as one token, while I split it into ప్రతి + రోజు. The tool also keeps కాలేజీకి and ఇంటికి as one token, while I split their suffixes. Both my manual version and the tool separate punctuation from the words. The manual version is more detailed because it separates some suffixes.

#Q5.3 — Multiword Expressions

Some Telugu MWEs are:

1. ప్రతి రోజు — every day
2. చాలా బాగా — very well
3. ఇంటి దగ్గర — near the house

These can be treated as single units because their words commonly occur together and express one combined meaning.
#Q5.4 — Reflection
The hardest part of Telugu tokenization is separating suffixes and word parts correctly. English is usually easier because spaces often show where words begin and end. Telugu words can contain more information inside a single word, so space-based tokenization is not always enough. Punctuation also needs to be separated from the words. MWEs can make tokenization harder because several words can work together to express one meaning. Overall, Telugu tokenization needs more attention to word structure than simple English space-based tokenization.
